# 퍼셉트론

In [ ]:
# 퍼셉트론(인공 신경망의 최소 단위)
# 입력(feature)과 가중치를 내적해서 bias를 더한 후,
# activation 함수(비선형 함수)를 통과시켜 결과값을 도출
# activate(x1*w1 + x2*w2 + ... + xn*wn + b)
#
# activation의 역할: 층을 쌓아도 선형으로 뭉개지지 않게 "비선형성"을 부여하는 것
# (분류/회귀 여부는 activation이 아니라 출력층 activation 종류 + loss 함수가 결정)
# 
# 다층 퍼셉트론: 퍼셉트론(내적+bias+activation)의 출력을
# 다음 층의 새로운 weight로 다시 조합해서, 
# 그 결과를 또 다른 퍼셉트론(activation)에 통과시키는 걸 n번 반복하는 구조

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# 데이터 로드 (이미 익숙한 데이터)
iris = load_iris()
X, y = iris.data, iris.target   # X: (150, 4) - 꽃잎/꽃받침 길이너비 4개 feature
                                # y: (150,)  - 품종 0,1,2 세 클래스

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ---- 여기가 핵심: 구조를 결정하는 부분 ----
clf = MLPClassifier(
  hidden_layer_sizes=(10, 10),
  # 히든층 2개, 각 층에 노드(퍼셉트론) 10개씩 병렬로 배치
  # 각 노드는 학습을 통해 weight/bias를 스스로 찾아냄
  # (AND/OR처럼 미리 정해진 역할이 있는 게 아니라, 자유롭게 학습됨)
  max_iter=1000,                 # 학습 반복 횟수 (역전파를 몇 번 돌릴지)
  random_state=42
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("정확도:", accuracy_score(y_test, y_pred))

정확도: 0.9333333333333333


# tensorflow
# Keras

In [ ]:
# TensorFlow: 텐서 연산 + 자동 미분을 GPU에서 처리하는 딥러닝 엔진
# Keras: 그 위에서 Sequential + Dense + compile + fit로 신경망을 쉽게 짤 수 있게 해주는 고수준 인터페이스

## 딥러닝 기초 - PyTorch에서 재확인할 개념 목록

### Optimizer 세부 종류
- **Momentum**: 이전 이동 방향의 관성을 유지해서 흔들림을 줄이고 빠르게 수렴
- **Adagrad**: weight마다 학습률을 자동으로 줄여나감 (너무 오래 학습하면 학습률이 지나치게 작아지는 단점)
- **RMSprop**: Adagrad의 단점을 보완 (최근 gradient에 더 가중치를 둠)
- **Adam**: momentum + RMSprop을 합친 것, 실무 기본값

### CNN
- **Conv2D**: 작은 필터를 이미지 위에 슬라이딩시키며 지역 패턴(모서리, 곡선 등)을 추출, 공간적 위치 관계를 보존
- **MaxPool2D**: 특징 맵을 축소해서 계산량을 줄이고 위치 변화에 덜 민감하게 만듦
- **Dense(MLP)와 차이**: Dense는 이미지를 그냥 숫자 나열로 취급(공간 구조 무시), CNN은 그 구조를 보존

### RNN / LSTM / GRU
- **RNN**: `state_new = tanh(w·x + u·state_old + b)` — 직전 출력을 현재 계산에 재사용해서 순서 있는 데이터 처리
- **기울기 소실(시간축)**: 문장/시퀀스가 길어지면 tanh 미분을 계속 곱하며 gradient가 사라짐
- **LSTM**: 게이트로 "기억을 오래 유지할지 잊을지" 결정해서 기울기 소실 완화
- **GRU**: LSTM을 단순화한 경량 버전, 성능은 비슷하되 계산은 가벼움

### 과적합 방지
- **L1 정규화**: weight 절댓값 합을 loss에 더해 일부 weight를 0으로 만듦 (sparse)
- **L2 정규화**: weight 제곱합을 loss에 더해 weight를 골고루 작게 줄임 (weight decay)
- **Dropout**: 학습 중 일부 노드를 무작위로 꺼서 특정 노드 의존을 막음 (앙상블과 비슷한 효과)
- **BatchNormalization**: 매 층/매 배치마다 입력 분포를 정규화 → 초기화 의존도 감소 + 학습속도 향상 + 과적합 억제

### 가중치 초기화
- 초기값이 너무 크면 → activation 양 끝단(0/1)에 쏠려 기울기 소실 시작
- 초기값이 너무 작으면 → 값이 중앙에 뭉개져 표현력 손실
- **Xavier (Glorot)**: 표준편차 `1/√n`, sigmoid/tanh에 적합
- **He**: 표준편차 `√(2/n)`, relu에 적합 (relu가 음수를 죽이므로 분산 보존을 위해 계수 2 필요)

### 텍스트 전처리
- **원-핫 인코딩**: 각 단어/클래스를 독립 벡터로 표현 (순서 관계로 오인되는 것 방지)
- **Word2Vec (CBOW / Skip-Gram)**: 단어를 저차원 실수 벡터로 학습, 의미가 비슷한 단어는 벡터 공간에서도 가까움
- **Embedding 레이어**: Word2Vec 아이디어를 신경망 학습 과정에 자동으로 포함시킨 것

In [ ]:
"""
=====================================================
예제 1. 기본 MLP 학습 파이프라인 (Keras)
- Sequential, Dense, activation, compile, fit 흐름
=====================================================
"""
import tensorflow as tf
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu'),   # 히든층: activation 필수 (비선형성)
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')  # 출력층: 다중분류라 softmax
])

# loss/optimizer/metrics를 미리 정해두는 단계 (아직 학습 안 함)
model.compile(
    loss='sparse_categorical_crossentropy',  # 라벨이 정수(0,1,2)일 때
    optimizer='adam',                          # 실무 기본값 (momentum + 적응적 학습률)
    metrics=['accuracy']
)

model.summary()

# epochs: 전체 데이터를 몇 바퀴 도는지
# batch_size: 한 번 업데이트에 데이터를 얼마나 보는지
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=16,
    validation_data=(X_test, y_test),
    verbose=0
)

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print("정확도:", acc)

In [ ]:
"""
=====================================================
예제 2. 과적합 방지 (L1/L2 정규화 + Dropout + BatchNorm)
- 실무에서 거의 항상 같이 쓰는 3종 세트
=====================================================
"""
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Dense(
        128,
        kernel_regularizer=tf.keras.regularizers.l2(0.001)  # L2: weight 크기 억제
    ),
    tf.keras.layers.BatchNormalization(),   # Dense 직후, Activation 직전
    tf.keras.layers.Activation('relu'),

    tf.keras.layers.Dense(
        64,
        kernel_regularizer=tf.keras.regularizers.l2(0.001)
    ),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),

    tf.keras.layers.Dropout(0.3),  # 마지막 히든층-출력층 사이 한 곳만

    tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
"""
=====================================================
예제 3. 가중치 초기화 지정 (He / Xavier)
- relu엔 He, sigmoid/tanh엔 Xavier(Glorot)가 실무 기본값
=====================================================
"""
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Dense(
        128, activation='relu',
        kernel_initializer='he_normal'        # relu용
    ),
    tf.keras.layers.Dense(
        64, activation='tanh',
        kernel_initializer='glorot_normal'    # Xavier의 다른 이름, tanh/sigmoid용
    ),
    tf.keras.layers.Dense(10, activation='softmax')
])

# 참고: activation='relu'만 쓰면 Keras가 기본적으로 glorot_uniform을 씀
# he_normal을 명시해주는 게 relu에는 더 적합

In [ ]:
"""
=====================================================
예제 4. CNN (이미지 분류) - Conv2D, MaxPool2D, BatchNorm
=====================================================
"""
import tensorflow as tf
import numpy as np

mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = np.expand_dims(x_train / 255.0, axis=-1)  # (N,28,28) -> (N,28,28,1) 채널 추가
x_test = np.expand_dims(x_test / 255.0, axis=-1)

model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), padding='same', input_shape=(28, 28, 1)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.MaxPool2D(pool_size=(2, 2)),

    tf.keras.layers.Conv2D(64, (3, 3), padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.MaxPool2D(pool_size=(2, 2)),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history = model.fit(x_train, y_train, epochs=5, batch_size=128,
                     validation_data=(x_test, y_test), verbose=0)

In [ ]:
"""
=====================================================
예제 5. RNN 계열 (시계열/텍스트) - Embedding, LSTM
- 예지보전 센서 시계열에도 구조적으로 재사용 가능한 패턴
  (Embedding 대신 센서값 그대로, 시퀀스 형태만 유지하면 됨)
=====================================================
"""
import tensorflow as tf

vocab_size = 1000
max_len = 300

# 텍스트 예시 (Embedding 사용)
text_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 32, input_length=max_len),  # 정수 인덱스 -> 학습가능한 벡터
    tf.keras.layers.LSTM(32),        # SimpleRNN보다 장기 의존성 처리 우수
    tf.keras.layers.Dense(1, activation='sigmoid')  # 이진분류
])
text_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])


# 시계열 센서 데이터 예시 (Embedding 없이, 수치 시퀀스 그대로)
# 입력 shape: (샘플 수, 타임스텝 수, 센서 개수)
n_timesteps = 50
n_sensors = 5

sensor_model = tf.keras.Sequential([
    tf.keras.layers.LSTM(64, input_shape=(n_timesteps, n_sensors), return_sequences=False),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')  # 예: 고장(1)/정상(0)
])
sensor_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [2]:
"""
=====================================================
예제 6. 원-핫 인코딩된 라벨 사용 시 loss 차이
- to_categorical 쓸 때는 categorical_crossentropy
- 정수 라벨 그대로 쓸 때는 sparse_categorical_crossentropy
=====================================================
"""
import tensorflow as tf
from tensorflow.keras.utils import to_categorical

y_train_int = [0, 1, 2, 1, 0]           # 정수 라벨
y_train_onehot = to_categorical(y_train_int, num_classes=3)  # [[1,0,0],[0,1,0],...]

# 정수 라벨 그대로 쓸 경우
model_a = tf.keras.Sequential([tf.keras.layers.Dense(3, activation='softmax')])
model_a.compile(loss='sparse_categorical_crossentropy', optimizer='adam')

# 원-핫 라벨 쓸 경우
model_b = tf.keras.Sequential([tf.keras.layers.Dense(3, activation='softmax')])
model_b.compile(loss='categorical_crossentropy', optimizer='adam')
print("성공")

성공
